In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

RNG = np.random.default_rng(42)

In [2]:
# PR
N_PRODUCTS = 100
CATEGORIES = ["Electronics", "Apparel", "Home & Living",
              "Beauty & Personal Care", "Sports & Outdoors", "Food & Beverage"]

CATEGORY_PRICE_RANGE = {
    "Electronics": (40, 900),
    "Apparel": (8, 150),
    "Home & Living": (15, 400),
    "Beauty & Personal Care": (5, 120),
    "Sports & Outdoors": (10, 350),
    "Food & Beverage": (3, 60),
}

ADJ = ["Pro", "Max", "Lite", "Plus", "Essential", "Classic", "Ultra", "Prime",
       "Everyday", "Signature", "Compact", "Advanced", "Basic", "Deluxe", "Smart"]
NOUN_BY_CAT = {
    "Electronics": ["Wireless Earbuds", "Bluetooth Speaker", "Smartwatch", "Power Bank",
                    "LED Desk Lamp", "Laptop Stand", "USB-C Hub", "Home Router", "Webcam", "Tablet Sleeve"],
    "Apparel": ["Cotton T-Shirt", "Denim Jacket", "Running Shorts", "Wool Sweater",
                "Rain Jacket", "Casual Shirt", "Yoga Pants", "Baseball Cap", "Ankle Socks", "Winter Scarf"],
    "Home & Living": ["Ceramic Mug Set", "Throw Blanket", "Storage Basket", "Wall Clock",
                      "Scented Candle", "Cutlery Set", "Bath Towel Set", "Cushion Cover", "Table Lamp", "Photo Frame"],
    "Beauty & Personal Care": ["Face Moisturizer", "Shampoo", "Hair Serum", "Body Lotion",
                               "Sunscreen SPF50", "Lip Balm Set", "Facial Cleanser", "Deodorant", "Hand Cream", "Perfume"],
    "Sports & Outdoors": ["Yoga Mat", "Water Bottle", "Resistance Bands", "Hiking Backpack",
                          "Camping Tent", "Cycling Gloves", "Football", "Jump Rope", "Sports Towel", "Trekking Pole"],
    "Food & Beverage": ["Herbal Tea Box", "Roasted Coffee Beans", "Protein Bar Pack", "Granola Mix",
                        "Fruit Preserve Jar", "Sparkling Water Pack", "Trail Mix Pouch", "Honey Jar", "Spice Set", "Cookie Box"],
}

products = []
for i in range(1, N_PRODUCTS + 1):
    category = CATEGORIES[(i - 1) % len(CATEGORIES)]
    low, high = CATEGORY_PRICE_RANGE[category]
    standard_price = round(RNG.uniform(low, high), 2)
    # cost as a % of price, varies by category, with noise
    base_cost_ratio = RNG.uniform(0.45, 0.75)
    cost_price = round(standard_price * base_cost_ratio, 2)
    margin_pct = round((standard_price - cost_price) / standard_price * 100, 2)
    noun = RNG.choice(NOUN_BY_CAT[category])
    adj = RNG.choice(ADJ)
    name = f"{adj} {noun}"
    active_status = "Active" if RNG.random() < 0.90 else "Discontinued"
    products.append({
        "product_id": f"P{i:04d}",
        "product_name": name,
        "category": category,
        "standard_price": standard_price,
        "cost_price": cost_price,
        "margin_percentage": margin_pct,
        "active_status": active_status,
    })

products_df = pd.DataFrame(products)

underperforming_idx = RNG.choice(products_df.index, size=8, replace=False)
products_df.loc[underperforming_idx, "cost_price"] = (
    products_df.loc[underperforming_idx, "standard_price"] * RNG.uniform(0.85, 0.95, size=8)
).round(2)
products_df.loc[underperforming_idx, "margin_percentage"] = (
    (products_df.loc[underperforming_idx, "standard_price"] - products_df.loc[underperforming_idx, "cost_price"])
    / products_df.loc[underperforming_idx, "standard_price"] * 100
).round(2)

In [3]:
# Customers
N_CUSTOMERS = 2000
CUSTOMER_TYPES = RNG.choice(["Individual", "SME", "Corporate"], size=N_CUSTOMERS, p=[0.70, 0.20, 0.10])
SEGMENTS = RNG.choice(["Premium", "Standard", "Budget", "New"], size=N_CUSTOMERS, p=[0.15, 0.45, 0.25, 0.15])
ACQ_CHANNELS = RNG.choice(["Referral", "Online", "Walk-in", "Sales Team"], size=N_CUSTOMERS, p=[0.20, 0.40, 0.25, 0.15])
LOCATIONS = RNG.choice(["Urban", "Suburban", "Other"], size=N_CUSTOMERS, p=[0.55, 0.35, 0.10])
AGE_BANDS = ["18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
BIZ_SIZES = ["Small (1-10)", "Medium (11-50)", "Large (51-200)", "Enterprise (200+)"]

start_window = datetime(2022, 1, 1)
end_window = datetime(2025, 11, 1)
window_days = (end_window - start_window).days

customers = []
for i in range(1, N_CUSTOMERS + 1):
    ctype = CUSTOMER_TYPES[i - 1]
    since_date = start_window + timedelta(days=int(RNG.integers(0, window_days)))
    age_band = RNG.choice(AGE_BANDS) if ctype == "Individual" else None
    biz_size = RNG.choice(BIZ_SIZES) if ctype != "Individual" else None
    customers.append({
        "customer_id": f"C{i:05d}",
        "customer_type": ctype,
        "segment": SEGMENTS[i - 1],
        "acquisition_channel": ACQ_CHANNELS[i - 1],
        "customer_since": since_date.strftime("%Y-%m-%d"),
        "location_category": LOCATIONS[i - 1],
        "age_band": age_band if age_band else "",
        "business_size": biz_size if biz_size else "",
    })

customers_df = pd.DataFrame(customers)

churn_flag = RNG.random(N_CUSTOMERS) < 0.15
customers_df["_churned"] = churn_flag  

In [4]:
# Campaigns
N_CAMPAIGNS = 20
OFFER_TYPES = ["Discount", "Bundle", "Cashback", "Seasonal Offer"]
CAMPAIGN_THEMES = ["New Year Sale", "Summer Splash", "Mid-Year Clearance", "Back to School",
                   "Festive Bonanza", "Flash Weekend", "Loyalty Rewards", "End of Season",
                   "Anniversary Sale", "Black Friday Deal", "Cyber Week Special", "Spring Refresh",
                   "Monsoon Offer", "Diwali Delight", "Christmas Cheer", "Valentine Special",
                   "Independence Day Sale", "Harvest Fest", "Clearance Blowout", "Winter Warmth"]

campaigns = []
overall_start = datetime(2024, 1, 1)
for i in range(1, N_CAMPAIGNS + 1):
    c_start = overall_start + timedelta(days=int(RNG.integers(0, 700)))
    duration = int(RNG.integers(7, 30))
    c_end = c_start + timedelta(days=duration)
    campaigns.append({
        "campaign_id": f"CMP{i:02d}",
        "campaign_name": CAMPAIGN_THEMES[i - 1],
        "start_date": c_start.strftime("%Y-%m-%d"),
        "end_date": c_end.strftime("%Y-%m-%d"),
        "offer_type": RNG.choice(OFFER_TYPES),
        "campaign_cost": round(RNG.uniform(2000, 40000), 2),
    })

campaigns_df = pd.DataFrame(campaigns)
campaigns_df["_start"] = pd.to_datetime(campaigns_df["start_date"])
campaigns_df["_end"] = pd.to_datetime(campaigns_df["end_date"])


In [5]:
# Sales Transactions 
N_TRANSACTIONS = 50000
BRANCHES = [f"BR{str(i).zfill(2)}" for i in range(1, 9)]  
CHANNELS = ["Online", "Branch", "Dealer", "Direct Sales", "Partner"]
CHANNEL_PROB = [0.35, 0.25, 0.15, 0.15, 0.10]

MONTHS = pd.date_range("2024-01-01", "2025-12-31", freq="MS")

SEASONAL_WEIGHT = {
    1: 0.95, 2: 0.85, 3: 1.00, 4: 1.00, 5: 0.95, 6: 0.85,
    7: 0.90, 8: 0.95, 9: 1.00, 10: 1.10, 11: 1.35, 12: 1.45,
}
month_weights = np.array([SEASONAL_WEIGHT[m.month] for m in MONTHS], dtype=float)
month_weights = month_weights / month_weights.sum()

month_choices = RNG.choice(len(MONTHS), size=N_TRANSACTIONS, p=month_weights)
tx_dates = []
for idx in month_choices:
    m = MONTHS[idx]
    days_in_month = (pd.Timestamp(m.year, m.month, 1) + pd.offsets.MonthEnd(0)).day
    day = int(RNG.integers(1, days_in_month + 1))
    tx_dates.append(datetime(m.year, m.month, day))
tx_dates = pd.to_datetime(tx_dates)

active_customers = customers_df.loc[~customers_df["_churned"], "customer_id"].values
churned_customers = customers_df.loc[customers_df["_churned"], "customer_id"].values
churn_cutoff = datetime(2025, 3, 31)  

customer_ids_for_tx = []
for d in tx_dates:
    if d <= churn_cutoff:
        cust = RNG.choice(customers_df["customer_id"].values)
    else:
        if RNG.random() < 0.03:
            cust = RNG.choice(churned_customers)
        else:
            cust = RNG.choice(active_customers)
    customer_ids_for_tx.append(cust)
customer_ids_for_tx = np.array(customer_ids_for_tx)

active_products = products_df.loc[products_df["active_status"] == "Active", "product_id"].values

product_weights = RNG.pareto(a=2.0, size=len(active_products)) + 0.1
product_weights = product_weights / product_weights.sum()
product_ids_for_tx = RNG.choice(active_products, size=N_TRANSACTIONS, p=product_weights)

branch_ids_for_tx = RNG.choice(BRANCHES, size=N_TRANSACTIONS, p=[0.18, 0.15, 0.14, 0.13, 0.12, 0.10, 0.10, 0.08])
channels_for_tx = RNG.choice(CHANNELS, size=N_TRANSACTIONS, p=CHANNEL_PROB)
quantities = RNG.poisson(lam=2.2, size=N_TRANSACTIONS) + 1  # 1..~10

prod_lookup = products_df.set_index("product_id")[["standard_price", "cost_price"]]
unit_prices = prod_lookup.loc[product_ids_for_tx, "standard_price"].values
unit_costs = prod_lookup.loc[product_ids_for_tx, "cost_price"].values

unit_prices = unit_prices * RNG.uniform(0.95, 1.05, size=N_TRANSACTIONS)
unit_prices = np.round(unit_prices, 2)

base_discount = RNG.choice(
    [0, 5, 10, 15, 20],
    size=N_TRANSACTIONS,
    p=[0.30, 0.25, 0.20, 0.15, 0.10]
).astype(float)

campaign_id_for_tx = np.array([""] * N_TRANSACTIONS, dtype=object)
tx_dates_ts = pd.Series(tx_dates)
for _, camp in campaigns_df.iterrows():
    mask = (tx_dates_ts >= camp["_start"]) & (tx_dates_ts <= camp["_end"])
    mask = mask.values
 
    attribute_mask = mask & (RNG.random(N_TRANSACTIONS) < 0.40)
    campaign_id_for_tx[attribute_mask] = camp["campaign_id"]
    base_discount[attribute_mask] += RNG.uniform(5, 15, size=attribute_mask.sum())

excessive_mask = RNG.random(N_TRANSACTIONS) < 0.04
base_discount[excessive_mask] += RNG.uniform(15, 25, size=excessive_mask.sum())
discount_pct = np.clip(base_discount, 0, 60)
discount_pct = np.round(discount_pct, 2)

gross_revenue = np.round(quantities * unit_prices, 2)
net_revenue = np.round(gross_revenue * (1 - discount_pct / 100), 2)
total_cost = np.round(quantities * unit_costs, 2)
gross_margin = np.round(net_revenue - total_cost, 2)

cust_type_lookup = customers_df.set_index("customer_id")["customer_type"]
tx_cust_type = cust_type_lookup.loc[customer_ids_for_tx].values

payment_status = np.empty(N_TRANSACTIONS, dtype=object)
rand_pay = RNG.random(N_TRANSACTIONS)
is_business = np.isin(tx_cust_type, ["SME", "Corporate"])

overdue_thresh = np.where(is_business, 0.12, 0.05)
pending_thresh = overdue_thresh + np.where(is_business, 0.08, 0.05)
partial_thresh = pending_thresh + 0.03

payment_status[rand_pay < overdue_thresh] = "Overdue"
payment_status[(rand_pay >= overdue_thresh) & (rand_pay < pending_thresh)] = "Pending"
payment_status[(rand_pay >= pending_thresh) & (rand_pay < partial_thresh)] = "Partially Paid"
payment_status[rand_pay >= partial_thresh] = "Paid"

transaction_ids = [f"TXN{str(i).zfill(6)}" for i in range(1, N_TRANSACTIONS + 1)]

sales_df = pd.DataFrame({
    "transaction_id": transaction_ids,
    "transaction_date": tx_dates.strftime("%Y-%m-%d"),
    "customer_id": customer_ids_for_tx,
    "product_id": product_ids_for_tx,
    "branch_id": branch_ids_for_tx,
    "channel": channels_for_tx,
    "quantity": quantities,
    "unit_price": unit_prices,
    "discount_percentage": discount_pct,
    "gross_revenue": gross_revenue,
    "net_revenue": net_revenue,
    "cost": total_cost,
    "gross_margin": gross_margin,
    "payment_status": payment_status,
    "campaign_id": campaign_id_for_tx,
})

sales_df = sales_df.sort_values("transaction_date").reset_index(drop=True)

In [6]:
customers_df = customers_df.drop(columns=["_churned"])
campaigns_df = campaigns_df.drop(columns=["_start", "_end"])

In [8]:
# Save to CSV
import os
OUT_DIR = "C:/Users/SHINI/Revenue_Analytics_Project/data"
os.makedirs(OUT_DIR, exist_ok=True)

products_df.to_csv(f"{OUT_DIR}/products.csv", index=False)
customers_df.to_csv(f"{OUT_DIR}/customers.csv", index=False)
campaigns_df.to_csv(f"{OUT_DIR}/campaigns.csv", index=False)
sales_df.to_csv(f"{OUT_DIR}/sales_transactions.csv", index=False)

print("Generated:")
print(f"  products.csv            -> {len(products_df)} rows")
print(f"  customers.csv           -> {len(customers_df)} rows")
print(f"  campaigns.csv           -> {len(campaigns_df)} rows")
print(f"  sales_transactions.csv  -> {len(sales_df)} rows")
print(f"Date range: {sales_df['transaction_date'].min()} to {sales_df['transaction_date'].max()}")

Generated:
  products.csv            -> 100 rows
  customers.csv           -> 2000 rows
  campaigns.csv           -> 20 rows
  sales_transactions.csv  -> 50000 rows
Date range: 2024-01-01 to 2025-12-31
